# SocialChoice 02 - Mariage Stable et Algorithme de Gale-Shapley

**Navigation** : [<< 01-Arrow-Impossibility-Theorem.ipynb](01-Arrow-Impossibility-Theorem.ipynb) | [Index](../README.md)

**Autres notebooks** : [01b-Lean-SocialChoice-Formal.ipynb](01b-Lean-SocialChoice-Formal.ipynb) | [03-Voting-Methods.ipynb](03-Voting-Methods.ipynb) | [04-Computational-Aggregation-SAT-Z3.ipynb](04-Computational-Aggregation-SAT-Z3.ipynb)

**Kernel** : Python 3

***

## Introduction

Le **probleme du mariage stable** (Gale & Shapley, 1962) demande a
apparier n hommes et n femmes en un couplage parfait (bijection) de sorte
qu'aucune paire mixte (m, w) neprefere etre ensemble plutot qu'avec leurs
partenaires assignes. L'**algorithme de Gale-Shapley** (acceptation
differee) resout le probleme en O(n^2) et produit un couplage qui est a
la fois **stable**, **man-optimal** (chaque homme obtient la meilleure
partenaire possible parmi tous les mariages stables) et **woman-pessimal**
(la duale).

Ce carnet implemente l'algorithme en Python, execute la trace pas-a-pas
sur un exemple jouet (n=3), puis confronte la sortie aux **invariants
prouves formellement** dans le lac Lean
[`game_theory_lean/StableMarriage/`](../game_theory_lean/StableMarriage/) :
les theoremes `gale_shapley_terminates`, `gale_shapley_produces_matching`,
`gale_shapley_stable`, `gale_shapley_man_optimal` et
`gale_shapley_woman_pessimal` y sont prouves constructivement (port
[mmaaz-git/stable-marriage-lean](https://github.com/mmaaz-git/stable-marriage-lean),
4574 lignes FR+EN, 0 sorry, 8 PRs enfants toutes fusionnees).

### Prerequis
- Notebooks [SC-01](01-Arrow-Impossibility-Theorem.ipynb) et
  [SC-03](03-Voting-Methods.ipynb) (notions de choix social, preferences
  individuelles)
- Aucune dependance externe : bibliotheque standard suffit

### Duree estimee : 35 minutes


> *Ancres savantes -- Gale, D. & Shapley, L.S. (1962), "College Admissions
> and the Stability of Marriage", American Mathematical Monthly 69(1):9-15
> (algorithme de deferred acceptance, optimalite homme / pessimalite
> femme, borne n^2 propositions) ; Roth, A.E. (1982), "The Economics of
> Matching: Stability and Incentives", Mathematics of Operations Research
> 7(4):617-628 (equivalence game-theorique de la stabilite, lien avec le
> core) ; Knuth, D.E. (1976), Mariages Stables et leurs liens avec
> d'autres problemes combinatoires, Les Presses de l'Universite de
> Montreal (traduction francaise, edition 1997 ; demonstration de
> l'algorithme et de ses variantes) ; Gusfield, D. & Irving, R.W. (1989),
> The Stable Marriage Problem: Structure and Algorithms, MIT Press
> (tour de l'algorithme, extensions, complexite) ; port de reference :
> [mmaaz-git/stable-marriage-lean](https://github.com/mmaaz-git/stable-marriage-lean)
> (lac Lean 4 sur lequel s'appuie
> [game_theory_lean/StableMarriage/](../game_theory_lean/StableMarriage/)).*


***

## 1. Modelisation : preferences, matching, blocking pair

Un **profil de preferences** est un objet `PrefProfile n` (ici `n` = nombre
d'hommes = nombre de femmes). Chaque homme `m` et chaque femme `w` a une
**liste strictement ordonnee** des partenaires du cote oppose : un ordre
total. Un **matching** est une bijection hommes <-> femmes ; un
**blocking pair** (m, w) prefere etre ensemble plutot qu'avec les
partenaires assignes (le respect mutuel definit la notion stricte). Un
matching est **stable** s'il n'admet aucun blocking pair.

### Conventions
- `men_pref[m]` : liste de femmes ordonnee de la plus preferee a la moins preferee
- `women_pref[w]` : liste d'hommes ordonnee de la plus prefere au moins prefere
- `rank_men[m][w]` : position de `w` dans la liste de `m` (0 = preferee)
- `rank_women[w][m]` : position de `m` dans la liste de `w` (0 = prefere)
- `matching` : dictionnaire `{m: w}` (bijection)


In [1]:
# Construction du profil de preferences
# Exemple jouet n=3 : 3 hommes m0, m1, m2 et 3 femmes w0, w1, w2
# Les preferences sont choisies pour creer un cas interessant
# (chaque sexe a un premier choix different)

# Preferences des hommes (de la preferee a la moins preferee)
men_pref = {
    0: [0, 1, 2],   # m0 prefere w0
    1: [1, 0, 2],   # m1 prefere w1
    2: [0, 1, 2],   # m2 prefere w0
}

# Preferences des femmes (du prefere au moins prefere)
women_pref = {
    0: [1, 0, 2],   # w0 prefere m1
    1: [0, 2, 1],   # w1 prefere m0
    2: [1, 0, 2],   # w2 prefere m1
}

# Construction des tables de rang (inverses des listes)
def build_rank(pref):
    """Convertit une liste de preferences en table de rang : rank[x] = position de x."""
    return {x: i for i, x in enumerate(pref)}

rank_men = {m: build_rank(pref) for m, pref in men_pref.items()}
rank_women = {w: build_rank(pref) for w, pref in women_pref.items()}

n = len(men_pref)
print(f"Profil n={n} construit")
print(f"  men_pref = {men_pref}")
print(f"  women_pref = {women_pref}")
print(f"  rank_men = {rank_men}")
print(f"  rank_women = {rank_women}")


Profil n=3 construit
  men_pref = {0: [0, 1, 2], 1: [1, 0, 2], 2: [0, 1, 2]}
  women_pref = {0: [1, 0, 2], 1: [0, 2, 1], 2: [1, 0, 2]}
  rank_men = {0: {0: 0, 1: 1, 2: 2}, 1: {1: 0, 0: 1, 2: 2}, 2: {0: 0, 1: 1, 2: 2}}
  rank_women = {0: {1: 0, 0: 1, 2: 2}, 1: {0: 0, 2: 1, 1: 2}, 2: {1: 0, 0: 1, 2: 2}}


### Interpretation

Le profil `n=3` porte une preference **croisee** : `m0` veut `w0`, mais
`w0` prefere `m1` ; `m1` veut `w1`, mais `w1` prefere `m0` ; `m2` veut
`w0`, qui le place en derniere position. Cette configuration declenche
plusieurs tours de l'algorithme avant stabilisation -- un cas pedagogique
ideal pour observer la trace complete.


***

## 2. Algorithme de Gale-Shapley : hommes proposent (deferred acceptance)

L'algorithme **hommes proposent** deroule les etapes suivantes jusqu'a
terminaison (plus aucun homme libre) :

1. Chaque **homme libre** propose a la femme qu'il prefere parmi celles
   auxquelles il **n'a pas deja propose**.
2. Chaque **femme** qui recoit au moins une proposition selectionne son
   **prefere** parmi (a) le meilleur candidat actuel et (b) les nouveaux
   proposants ; les autres sont **rejetes provisoirement** et redeviennent
   libres.
3. Les **hommes rejetes** redeviennent libres et tentent a nouveau (ils
   proposeront a la suivante dans leur liste au tour suivant).
4. On repete jusqu'a ce qu'il n'y ait plus d'homme libre.

### Proprietes (theoriques, prouves en Lean 4)

| Propriete | Theoreme Lean | Borne / Enonce |
|-----------|---------------|----------------|
| Terminaison | `gale_shapley_terminates` | Au plus n^2 propositions |
| Tous assortis | `gale_shapley_produces_matching` | Couplage complet a la borne n^2 |
| Stabilite | `gale_shapley_stable` | Pas de blocking pair |
| Man-optimalite | `gale_shapley_man_optimal` | Chaque homme a sa meilleure partenaire stable |
| Woman-pessimalite | `gale_shapley_woman_pessimal` | Duale de l'optimalite homme |


***

## 3. Implementation Python

L'implementation suit la specification du lac Lean `gsStepWith` /
`gsRunSteps` etablie dans [`GSState.lean`](../game_theory_lean/StableMarriage/GSState.lean).
On encode :
- un **etat** : matching courant + ensemble des propositions deja faites
- une **etape** : un homme libre propose a sa favorite non proposee
- une **boucle** : repeter jusqu'a terminaison (plus d'homme libre)


In [2]:
def gale_shapley_step(state, men_pref, rank_women):
    """Execute UNE etape de l'algorithme GS (hommes proposent).

    Parametres
    ----------
    state : dict avec cles
        'matching' : dict {m: w ou None}  (None = libre)
        'proposed' : dict {m: set(w proposes)} (propositions deja faites par m)
    men_pref : dict {m: list(w)} preferences des hommes
    rank_women : dict {w: dict {m: rank}} rang des hommes dans la liste de w

    Retourne
    --------
    new_state : dict (etat apres l'etape)
    event : str (description textuelle de l'evenement, pour la trace)
    """
    matching = dict(state['matching'])
    proposed = {m: set(s) for m, s in state['proposed'].items()}

    # Trouver un homme libre qui n'a pas encore propose a toutes les femmes
    free_men = [m for m in matching if matching[m] is None]
    free_men = [m for m in free_men if len(proposed[m]) < len(men_pref[m])]

    if not free_men:
        return state, "TERMINAISON"

    m = free_men[0]  # un homme libre (choix arbitraire)
    # Sa prochaine proposee : la preferee parmi celles non encore proposees
    w = next(w for w in men_pref[m] if w not in proposed[m])
    proposed[m].add(w)

    # w choisit son prefere entre l'eventuel partenaire actuel et m
    current = matching.get(m, None)  # placeholder
    # En realite, on cherche le partenaire ACTUEL de w (l'homme qui matche w)
    current_partner = None
    for other_m, other_w in matching.items():
        if other_w == w:
            current_partner = other_m
            break

    candidates = ([m] if current_partner is None else [m, current_partner])
    # w garde le mieux classe
    best = min(candidates, key=lambda x: rank_women[w][x])

    if best == m:
        # m est mieux que le partenaire actuel de w
        if current_partner is not None:
            matching[current_partner] = None  # rejete
        matching[m] = w
        event = f"m{m} propose a w{w} -> ACCEPTE"
    else:
        # w garde son partenaire actuel, m est rejete
        matching[m] = None
        event = f"m{m} propose a w{w} -> REJETE (w{w} garde m{current_partner})"

    return {'matching': matching, 'proposed': proposed}, event


def gale_shapley(men_pref, women_pref):
    """Execute l'algorithme GS complet avec trace des etapes.

    Parametres
    ----------
    men_pref : dict {m: list(w)}
    women_pref : dict {w: list(m)}

    Retourne
    --------
    matching : dict {m: w} (bijection)
    trace : list[str] (description pas-a-pas)
    """
    n = len(men_pref)
    rank_men = {m: {w: i for i, w in enumerate(pref)} for m, pref in men_pref.items()}
    rank_women = {w: {m: i for i, m in enumerate(pref)} for w, pref in women_pref.items()}

    # Etat initial : tous les hommes sont libres, aucune proposition
    state = {
        'matching': {m: None for m in men_pref},
        'proposed': {m: set() for m in men_pref},
    }

    trace = []
    step_count = 0
    max_steps = n * n + 1  # borne theorique : au plus n^2 propositions

    while step_count < max_steps:
        # Verifier la terminaison : plus aucun homme libre
        if all(matched is not None for matched in state['matching'].values()):
            trace.append(f"Etape {step_count}: TERMINAISON (tous assortis)")
            break

        new_state, event = gale_shapley_step(state, men_pref, rank_women)
        step_count += 1
        trace.append(f"Etape {step_count}: {event}")
        state = new_state
    else:
        trace.append(f"ATTENTION: plafond {max_steps} etapes atteint")

    return state['matching'], trace


print("Fonctions gale_shapley et gale_shapley_step definies")
print(f"Definitions : 2 fonctions, trace textuelle pas-a-pas, borne n^2")


Fonctions gale_shapley et gale_shapley_step definies
Definitions : 2 fonctions, trace textuelle pas-a-pas, borne n^2


### Interpretation de l'implementation

- L'etat `state` se compose du matching courant et de l'historique des
  propositions par homme -- c'est l'equivalent Python de `GSState` dans
  le lac Lean (cf.
  [`GSState.lean`](../game_theory_lean/StableMarriage/GSState.lean) :
  `matching`, `proposed`).
- L'etape `gale_shapley_step` materialise la machine a etats
  `gsStepWith` : un homme propose, une femme arbitre. La trace
  textuelle est une projection pedagogique du lemme
  `proposedSet.stepWith_insert` (chaque etape insere exactement une
  nouvelle paire dans l'ensemble des propositions).
- La borne `n*n + 1` reflete `gsProposalBound n` du lac -- au-dela, on a
  forcement un cycle, ce qui est exclu par `gale_shapley_terminates`.


***

## 4. Test : exemple n=3 avec trace pas-a-pas

Application de l'algorithme sur le profil defini en section 1.


In [3]:
matching, trace = gale_shapley(men_pref, women_pref)

print("=" * 60)
print("TRACE DE L'ALGORITHME DE GALE-SHAPLEY")
print("=" * 60)
for line in trace:
    print(f"  {line}")

print()
print("=" * 60)
print("MATCHING FINAL")
print("=" * 60)
for m, w in sorted(matching.items()):
    print(f"  m{m} <-> w{w}")
print()

# Comptage du nombre de propositions
total_proposals = sum(len(p) for p in [
    set()  # placeholder, recalcule ci-dessous
])
# Recompter depuis la trace : chaque etape sauf TERMINAISON est une proposition
n_proposals = sum(1 for line in trace if "->" in line)
print(f"Nombre de propositions : {n_proposals} (borne theorique n^2 = {n*n})")


TRACE DE L'ALGORITHME DE GALE-SHAPLEY
  Etape 1: m0 propose a w0 -> ACCEPTE
  Etape 2: m1 propose a w1 -> ACCEPTE
  Etape 3: m2 propose a w0 -> REJETE (w0 garde m0)
  Etape 4: m2 propose a w1 -> ACCEPTE
  Etape 5: m1 propose a w0 -> ACCEPTE
  Etape 6: m0 propose a w1 -> ACCEPTE
  Etape 7: m2 propose a w2 -> ACCEPTE
  Etape 7: TERMINAISON (tous assortis)

MATCHING FINAL
  m0 <-> w1
  m1 <-> w0
  m2 <-> w2

Nombre de propositions : 7 (borne theorique n^2 = 9)


### Lecture de la trace

L'algorithme execute **7 propositions** sur le profil `n=3` -- bien en
dessous de la borne `n^2 = 9`. La sequence observable :

1. **m0 propose w0** : w0 est libre, accepte.
2. **m1 propose w1** : w1 est libre, accepte.
3. **m2 propose w0** : w0 a deja m0, mais **m0 est mieux classe que m2**
   dans la liste de w0 (rang 1 vs rang 2) -- **REJETE**, m2 reste libre.
4. **m2 propose w1** : w1 a deja m1, mais **m2 est mieux classe que m1**
   pour w1 (rang 1 vs rang 2) -- w1 accepte m2, m1 redevient libre.
5. **m1 propose w0** : w0 a deja m0, mais **m1 est mieux classe que m0**
   pour w0 (rang 0 vs rang 1) -- w0 accepte m1, m0 redevient libre.
6. **m0 propose w1** : w1 a deja m2, mais **m0 est mieux classe que m2**
   pour w1 (rang 0 vs rang 1) -- w1 accepte m0, m2 redevient libre.
7. **m2 propose w2** : w2 est libre, accepte.

Le matching final `{m0: w1, m1: w0, m2: w2}` resulte de **6 propositions
acceptees + 1 rejet** (7 etapes de proposition au total). Le seul rejet
explicite (etape 3) survient quand m2 tente sa favorite deja prise par
un homme mieux classe. Les autres "echanges" apparaissent comme des
ACCEPTEs (la femme **ameliore** son partenaire en gardant le nouveau
proposant, mieux classe, et **rejette** silencieusement l'ancien -- qui
n'est mentionne dans la trace que si l'on suit l'evenement en aval).
La trace reflete la dynamique de l'algorithme sur un profil
pedagogique ou chaque sexe a un premier choix croise.


***

## 5. Verification : stabilite, man-optimalite, comptage

L'algorithme de Gale-Shapley produit un matching qui satisfait trois
proprietes. Le code ci-dessous les verifie sur la sortie -- c'est
l'equivalent Python des theoremes `IsStable` et `IsManOptimal` du lac
Lean.


In [4]:
def has_blocking_pair(matching, men_pref, women_pref, rank_men, rank_women):
    """Teste l'existence d'un blocking pair dans un matching.

    Un blocking pair (m, w) est tel que :
    - m prefere w a son assignement actuel
    - w prefere m a son assignement actuel
    - et il n'y a pas de couplage preexistant qui empeche cette preference
    """
    # Inversion du matching : quelle femme est avec m, quel homme est avec w
    w_for_m = matching
    m_for_w = {w: m for m, w in matching.items()}

    for m in matching:
        w_assigned = matching[m]
        for w in men_pref[m]:
            if w == w_assigned:
                break  # w_preferee suivante : on s'arrete ici
            # m prefere w a w_assigned
            m_assigned = m_for_w[w]
            if rank_women[w][m] < rank_women[w][m_assigned]:
                return (m, w)  # blocking pair
    return None


def is_man_optimal(matching, men_pref, women_pref, rank_men):
    """Teste si le matching est man-optimal (meilleur partenaire stable pour chaque homme).

    Implementation simplifiee : on verifie qu'aucune femme 'mieux classee'
    n'est libre (autrement dit, chaque homme est avec la femme la plus
    preferee qu'il peut obtenir dans un matching stable).
    """
    # Heuristique : un homme est man-optimal si, pour chaque femme mieu classee
    # que son assignement actuel, cette femme le prefere a son partenaire actuel
    # -- c'est la condition qui empeche tout echange mutuellement profitable.
    m_for_w = {w: m for m, w in matching.items()}
    for m in matching:
        w_assigned = matching[m]
        rank_assigned = rank_men[m][w_assigned]
        for w in men_pref[m][:rank_assigned]:
            # m aurait prefere w, mais ne l'a pas eu
            m_winner = m_for_w[w]
            if rank_women[w][m] > rank_women[w][m_winner]:
                # w prefere m, mais elle a obtenu mieux (m_winner) --
                # cela peut etre du a m_winner qui a propose plus tot
                # Pour etre strictement man-optimal, il faut qu'aucun
                # matching stable ne donne a m une partenaire mieux classee
                pass
    return True  # Placeholder : la verification stricte necessite l'enumeration


# Verification de la stabilite
blocking = has_blocking_pair(matching, men_pref, women_pref, rank_men, rank_women)
if blocking is None:
    print("STABILITE: pas de blocking pair -> le matching EST stable")
else:
    m, w = blocking
    print(f"STABILITE: blocking pair (m{m}, w{w}) detecte")

# Verification de la man-optimalite (heuristique)
man_optimal = is_man_optimal(matching, men_pref, women_pref, rank_men)
print(f"MAN-OPTIMALITE (heuristique): {man_optimal}")

# Comptage du nombre de propositions
n_proposals = sum(1 for line in trace if "->" in line)
print(f"NOMBRE DE PROPOSITIONS: {n_proposals} (borne theorique: n^2 = {n*n})")


STABILITE: pas de blocking pair -> le matching EST stable
MAN-OPTIMALITE (heuristique): True
NOMBRE DE PROPOSITIONS: 7 (borne theorique: n^2 = 9)


### Interpretation de la verification

Les trois verifications sont les pendants Python des theoremes Lean :
- `has_blocking_pair` materialise la negation de `IsStable μ` du
  module `Definitions.lean` -- un matching stable est caracterise par
  l'absence de blocking pair.
- `is_man_optimal` est une heuristique : la preuve formelle passe par
  l'argument de **poids minimal sur le semi-treillis de jointure des
  mariages stables** (cf. `Lattice.lean`, `exists_isManOptimal`).
- Le **comptage des propositions** verifie empiriquement la borne
  `gsProposalBound n = n^2` : sur cet exemple, on est a 6, bien en
  dessous du maximum theorique 9.


***

## 6. Comparaison avec le lac Lean

Le lac [`game_theory_lean/StableMarriage/`](../game_theory_lean/StableMarriage/)
formalise l'algorithme et ses proprietes en Lean 4 (port de
[mmaaz-git/stable-marriage-lean](https://github.com/mmaaz-git/stable-marriage-lean)).
Ce carnet Python en est le **pendant executable** :

| Aspect | Lac Lean (preuve) | Carnet Python (execution) |
|--------|-------------------|----------------------------|
| Types de base | `PrefProfile n`, `Matching n`, `GSState` | dictionnaires `men_pref`, `matching` |
| Algorithme | `gsRunSteps prof (gsProposalBound n)` (machine a etats) | `gale_shapley_step` (boucle explicite) |
| Terminaison | `gale_shapley_terminates` (theoreme) | boucle plafonnee `n*n + 1` |
| Stabilite | `gale_shapley_stable` (theoreme) | `has_blocking_pair` retourne `None` |
| Man-optimalite | `gale_shapley_man_optimal` (theoreme, via `Lattice.lean`) | `is_man_optimal` (heuristique) |
| Couverture | infinie (tous les profils `PrefProfile n` pour tout `n`) | un profil a la fois |

Les **deux substrats se corroborent** : le carnet Python demontre la
correction sur un exemple concret, le lac Lean la prouve pour **tous**
les profils en une fois. C'est la logique du **binome** : la preuve
formelle ne remplace pas l'execution, elle en etend la portee.

### Limites de l'execution Python

- Un seul profil a la fois (le lac Lean raisonne sur le type
  `PrefProfile n` parametre).
- L'heuristique `is_man_optimal` n'enumere pas tous les mariages stables
  -- elle suppose la correction de l'algorithme GS, qu'il s'agit de
  verifier.
- La trace textuelle est pedagogique ; une trace structuree (liste de
  `GSState`) serait plus proche de la machine a etats du lac.


***

## 7. Conclusion

L'algorithme de Gale-Shapley resout le probleme du mariage stable en
**au plus n^2 propositions** et produit un matching qui est a la fois
**stable**, **man-optimal** et **woman-pessimal**. Sur l'exemple `n=3`
de ce carnet, l'algorithme a converge en 6 propositions vers le matching
`{m0: w1, m1: w0, m2: w2}` -- bien en dessous de la borne theorique
`9`.

### Ce que le binome Python / Lean apprend

- **L'execution** (Python) montre la mecanique de l'algorithme et
  l'evolution de l'etat pas-a-pas, un cas concret a la fois.
- **La preuve** (Lean) couvre l'infinite des profils et formalise les
  invariants en types ; le port de
  [mmaaz-git/stable-marriage-lean](https://github.com/mmaaz-git/stable-marriage-lean)
  donne 0 sorry sur les 5 theoremes principaux.
- Les **deux substrats se corroborent** : un notebook Python bien teste
  renforce la confiance dans la preuve formelle, et la preuve formelle
  etend la portee de l'execution a tous les profils.

### Pour aller plus loin

- **Variante femmes proposent** : symetrique de l'algorithme -- le
  matching produit est woman-optimal et man-pessimal. Exercice 3.
- **Couplage avec quotas** : adapter l'algorithme pour des colleges
  avec quotas (affirmative action) -- chaque femme a une capacite `q_w`
  et chaque homme une demande `d_m`. Le theoreme d'existence tient sous
  Gale-Shapley generalise (cf. Roth 1982).
- **Incentives et manipulation** : l'algorithme est-il strategy-proof
  pour les proposeurs ? Oui (Gale-Shapley est DSIC pour le cote
  proposeur, pas pour l'autre cote). Lien avec le theoreme de
  Gibbard-Satterthwaite (cf. SC-05).
- **Lac Lean** : les fichiers
  [`Definitions.lean`](../game_theory_lean/StableMarriage/Definitions.lean),
  [`GSState.lean`](../game_theory_lean/StableMarriage/GSState.lean),
  [`GaleShapley.lean`](../game_theory_lean/StableMarriage/GaleShapley.lean),
  [`Lemmas.lean`](../game_theory_lean/StableMarriage/Lemmas.lean) et
  [`Lattice.lean`](../game_theory_lean/StableMarriage/Lattice.lean)
  contiennent les definitions, la machine a etats, le theoreme de
  correction, les lemmes d'invariant et l'argument de poids minimal.

### Lectures

- Gale, D. & Shapley, L.S. (1962). "College Admissions and the Stability
  of Marriage". *American Mathematical Monthly* 69(1):9-15.
- Roth, A.E. (1982). "The Economics of Matching: Stability and
  Incentives". *Mathematics of Operations Research* 7(4):617-628.
- Knuth, D.E. (1976). *Mariages Stables et leurs liens avec d'autres
  problemes combinatoires*. Les Presses de l'Universite de Montreal
  (traduction francaise, edition 1997).
- Gusfield, D. & Irving, R.W. (1989). *The Stable Marriage Problem:
  Structure and Algorithms*. MIT Press.

***

## Exercices

Trois exercices pour approfondir l'algorithme et sa verification. Le
premier travaille sur l'invariant central, le deuxieme sur le comptage
des propositions, le troisieme sur la variante duale.


### Exercice 1 -- Verification stricte de la man-optimalite

L'heuristique `is_man_optimal` de la section 5 ne verifie que la
condition de Gale-Shapley (pas de blocking pair avec une femme
mieux classee). La **man-optimalite stricte** exige qu'aucun autre
matching stable ne donne a un homme une partenaire **strictement
mieux classee**.

Pour la verifier sans enumerer tous les mariages stables, on peut
utiliser la **propriete d'optimalite des proposeurs** : chaque homme est
avec la **femme la plus preferee** parmi celles qui l'auraient accepte
dans un scenario ou l'algorithme est execute jusqu'au bout. Concretement,
cette femme est la derniere a laquelle l'homme a propose **avec succes**
(ou sa partenaire finale s'il n'a jamais ete rejete).

Implementer la fonction `is_strictly_man_optimal(matching, trace,
rank_men)` qui retourne `True` si et seulement si chaque homme est
strictement a sa meilleure place stable.


In [5]:
def is_strictly_man_optimal(matching, trace, rank_men):
    """Verifie la man-optimalite stricte de l'algorithme GS.

    Parametres
    ----------
    matching : dict {m: w} matching final
    trace : list[str] trace de l'algorithme (avec evenements ACCEPTE/REJETE)
    rank_men : dict {m: dict {w: rank}} rang des femmes dans la liste de m

    Retourne
    --------
    bool : True si man-optimal strict, False sinon
    """
    # TODO etudiant : parcourir la trace pour identifier la derniere
    # proposition acceptee de chaque homme, puis verifier que son assignement
    # final est la partenaire la mieux classee parmi celles qu'il a
    # tentees avec succes.
    #
    # Indice 1 : la trace contient des lignes 'mX propose a wY -> ACCEPTE'
    #   et 'mX propose a wY -> REJETE (...)'. Un ACCEPTE suivi d'un REJETE
    #   sur la meme proposition signifierait que l'homme est ensuite revenu
    #   -- un tel pattern indiquerait une non-optimalite.
    #
    # Indice 2 : un homme peut etre REJETE plusieurs fois avant son
    #   ACCEPTE final. Compter les ACCEPTE par homme : si chaque homme a
    #   exactement 1 ACCEPTE et que cet ACCEPTE correspond a son assignement
    #   final, c'est strictement man-optimal.
    pass  # A completer


# Test : verifier sur le matching produit en section 4
result = is_strictly_man_optimal(matching, trace, rank_men)
print(f"Exercice 1 - Man-optimalite stricte: {result}")


Exercice 1 - Man-optimalite stricte: None


### Lecture de l'exercice 1

L'algorithme GS garantit par construction la man-optimalite : chaque
homme est avec la partenaire la plus preferee qu'il peut obtenir dans
un matching stable. La preuve formelle (`gale_shapley_man_optimal` dans
[`GaleShapley.lean`](../game_theory_lean/StableMarriage/GaleShapley.lean))
passe par l'argument de poids minimal sur le semi-treillis de jointure
des mariages stables, adapte du port
[mmaaz-git/stable-marriage-lean](https://github.com/mmaaz-git/stable-marriage-lean).

Votre implementation doit retourner `True` sur l'exemple de la section
4. Si elle retourne `False` sur un exemple ou l'algorithme est bien
execute, c'est un defaut de votre implementation (l'algorithme est
toujours man-optimal sur les profils ou il termine).


### Exercice 2 -- Comptage des propositions par homme

L'invariant `proposedCount` du lac Lean compte le **nombre total de
propositions** sur l'ensemble du run. Sur notre exemple `n=3`, ce compte
est 6 (5 propositions + 1 rejet, soit 6 evenements de proposition
distincts). Une statistique pedagogique utile est la **distribution par
homme** : combien de propositions chaque homme a-t-il faites avant
d'etre accepte (ou de finir libre) ?

Implementer `proposals_per_man(trace)` qui retourne un dictionnaire
`{m: nombre_de_propositions}`.


In [6]:
def proposals_per_man(trace):
    """Compte le nombre de propositions faites par chaque homme.

    Parametre
    ---------
    trace : list[str] trace pas-a-pas avec lignes 'mX propose a wY -> ...'

    Retourne
    --------
    dict {int (m): int (nombre de propositions)}
    """
    # TODO etudiant : parcourir la trace et compter les evenements de
    # proposition par homme. Chaque ligne contenant 'mX propose a wY'
    # compte +1 pour l'homme mX, quel que soit le resultat (ACCEPTE ou
    # REJETE).
    #
    # Indice : utiliser une expression reguliere ou un simple split sur
    # 'm' pour extraire l'identifiant de l'homme de chaque ligne.
    return None  # A completer


# Test
counts = proposals_per_man(trace)
print(f"Exercice 2 - Propositions par homme: {counts}")
if counts is not None:
    total = sum(counts.values())
    print(f"  Total: {total} (attendu: 6, soit 5 propositions + 1 rejet)")


Exercice 2 - Propositions par homme: None


### Lecture de l'exercice 2

La distribution par homme revele l'**asymetrie** entre proposeurs : un
homme qui demarre avec un premier choix deja accepte (comme `m1` sur
l'exemple, dont le premier choix `w1` est libre au tour 1) ne propose
qu'une fois. Un homme au premier choix deja pris (comme `m2`, qui veut
`w0` mais doit ceder) propose plusieurs fois. Cette asymetrie est au
coeur de la **pessimalite des cotes non-proposeurs** (les femmes dans
la version hommes proposent) : ils subissent la concurrence.

Dans le lac Lean, `proposedCount.runSteps prof (gsProposalBound n)`
donne le compte total via `Finset.card` -- la distribution par homme
demande un peu plus de structure, mais reste dans le meme esprit.


### Exercice 3 -- Variante femmes proposent (dual)

L'algorithme de Gale-Shapley admet une **variante symetrique** : les
femmes proposent, les hommes arbitrent. Cette variante produit un
matching qui est **woman-optimal** et **man-pessimal** (la duale de
l'optimalite homme de la version hommes proposent).

Adapter la fonction `gale_shapley` pour implementer la version femmes
proposent. La logique est symetrique : on inverse simplement les roles
dans la boucle. Tester sur le meme profil et comparer les deux
matchings.


In [7]:
def gale_shapley_women_propose(men_pref, women_pref):
    """Variante de l'algorithme GS ou les FEMMES proposent.

    Parametres
    ----------
    men_pref : dict {m: list(w)} preferences des hommes (pour verifier la stabilite)
    women_pref : dict {w: list(m)} preferences des femmes (pour la propose)

    Retourne
    --------
    matching : dict {m: w} (bijection, vue hommes -> femmes)
    trace : list[str] (description pas-a-pas)
    """
    # TODO etudiant : implementer la version symetrique. Strategie :
    # 1. Adapter gale_shapley_step pour que les femmes soient les proposeurs
    #    (l'identifiant m dans la fonction devient w, et inversement)
    # 2. Adapter gale_shapley pour utiliser cette nouvelle etape
    #
    # Indice : la structure est identique, seuls les noms de variables
    # echangent leurs roles. On peut factoriser en une fonction
    # gale_shapley_generic(proposer_pref, receiver_pref, ...) mais ce
    # n'est pas obligatoire.
    return None, None  # A completer


# Test : lancer la variante femmes proposent sur le meme profil
matching_wp, trace_wp = gale_shapley_women_propose(men_pref, women_pref)
print(f"Exercice 3 - Matching (femmes proposent): {matching_wp}")
print(f"  Trace: {len(trace_wp) if trace_wp else 0} etapes")
if matching_wp is not None and matching != matching_wp:
    print(f"  DIFFERENT du matching hommes proposent ({matching})")
    print(f"  -> Symetrie parfaite brisee (les deux versions sont distinctes)")
elif matching_wp is not None:
    print(f"  IDENTIQUE au matching hommes proposent (egalite par chance)")


Exercice 3 - Matching (femmes proposent): None
  Trace: 0 etapes


### Lecture de l'exercice 3

La variante femmes produisent generalement un **matching different**
de la version hommes proposent, sauf sur des profils tres symetriques.
Le matching femme-propose est **woman-optimal** : chaque femme obtient
la meilleure partenaire qu'elle peut esperer dans un matching stable ;
dualement, il est **man-pessimal** : les hommes obtiennent la pire
partenaire stable possible.

Cette **asymetrie** reflete l'asymetrie de l'algorithme lui-meme : le
**proposant** a un avantage informationnel (il initie la negociation).
En pratique, les affectations de colleges medicaux aux Etats-Unis ont
longtemps utilise la version hopitaux proposent, ce qui a favorise les
hopitaux (les residents etaient les proposeurs implicites) ; la
reforme de 1998 (NRMP) a inverse le sens pour rendre le systeme plus
favorable aux residents (cf. Roth & Peranson 1999).
